# Netflix Viewing Analytics Pipeline (ETL-style)

**Goal:** build a daily report showing total viewing activity, most watched movies, active users, and average ratings, using `Movie`, `User`, and `WatchHistory` classes.

**Pipeline:** Extract → Transform → Load (report)

## 1. Imports and domain classes

In [1]:
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import date


@dataclass
class Movie:
    movie_id: int
    title: str
    genre: str


@dataclass
class User:
    user_id: int
    name: str


@dataclass
class WatchHistory:
    user_id: int
    movie_id: int
    watched_on: date
    minutes: int
    rating: float | None = None  # 1-5, optional

## 2. Extract
Load raw records. In a real pipeline this would read CSV files, a database, or an API. The history includes deliberately bad rows (unknown movie, negative minutes, rating of 9) to test the cleaning step.

In [2]:
def extract():
    movies = [
        {"movie_id": 1, "title": "Inception", "genre": "Sci-Fi"},
        {"movie_id": 2, "title": "The Crown", "genre": "Drama"},
        {"movie_id": 3, "title": "Stranger Things", "genre": "Horror"},
    ]
    users = [
        {"user_id": 10, "name": "Aisha"},
        {"user_id": 11, "name": "Ben"},
        {"user_id": 12, "name": "Chen"},
    ]
    history = [
        {"user_id": 10, "movie_id": 1, "watched_on": "2026-11-04", "minutes": 120, "rating": 5},
        {"user_id": 10, "movie_id": 3, "watched_on": "2026-11-04", "minutes": 45, "rating": 4},
        {"user_id": 11, "movie_id": 1, "watched_on": "2026-11-04", "minutes": 95, "rating": 4},
        {"user_id": 12, "movie_id": 2, "watched_on": "2026-11-04", "minutes": 50, "rating": None},
        {"user_id": 11, "movie_id": 99, "watched_on": "2026-11-04", "minutes": 30, "rating": 3},   # bad movie id
        {"user_id": 12, "movie_id": 1, "watched_on": "2026-11-04", "minutes": -5, "rating": 4},    # bad minutes
        {"user_id": 10, "movie_id": 2, "watched_on": "2026-11-03", "minutes": 60, "rating": 5},    # other day
        {"user_id": 11, "movie_id": 3, "watched_on": "2026-11-04", "minutes": 40, "rating": 9},    # bad rating
    ]
    return movies, users, history

## 3. Transform
Build objects and reject invalid records (unknown IDs, non-positive minutes, ratings outside 1–5).

In [3]:
def transform(raw_movies, raw_users, raw_history):
    movies = {m["movie_id"]: Movie(**m) for m in raw_movies}
    users = {u["user_id"]: User(**u) for u in raw_users}

    clean, rejected = [], []
    for r in raw_history:
        valid = (
            r["movie_id"] in movies
            and r["user_id"] in users
            and r["minutes"] > 0
            and (r["rating"] is None or 1 <= r["rating"] <= 5)
        )
        if valid:
            clean.append(WatchHistory(
                r["user_id"], r["movie_id"],
                date.fromisoformat(r["watched_on"]), r["minutes"], r["rating"],
            ))
        else:
            rejected.append(r)
    return movies, users, clean, rejected

## 4. Load: build the daily report

In [4]:
def build_report(movies, users, history, report_day):
    today = [h for h in history if h.watched_on == report_day]

    views = Counter(h.movie_id for h in today)
    ratings = defaultdict(list)
    for h in today:
        if h.rating is not None:
            ratings[h.movie_id].append(h.rating)

    return {
        "date": report_day.isoformat(),
        "total_minutes": sum(h.minutes for h in today),
        "total_views": len(today),
        "most_watched": [(movies[mid].title, n) for mid, n in views.most_common(3)],
        "active_users": sorted({users[h.user_id].name for h in today}),
        "avg_ratings": {movies[mid].title: round(sum(r) / len(r), 2)
                        for mid, r in ratings.items()},
    }

In [5]:
def print_report(rep, n_rejected):
    print(f"=== Daily Netflix Report: {rep['date']} ===")
    print(f"Total viewing activity : {rep['total_views']} views, {rep['total_minutes']} minutes")
    print("Most watched movies    :", ", ".join(f"{t} ({n})" for t, n in rep["most_watched"]))
    print(f"Active users ({len(rep['active_users'])})     :", ", ".join(rep["active_users"]))
    print("Average ratings        :", rep["avg_ratings"])
    print(f"(Data quality: {n_rejected} invalid records rejected)")

## 5. Run the pipeline

In [6]:
def run(report_day=date(2026, 11, 4)):
    raw = extract()
    movies, users, history, rejected = transform(*raw)
    print_report(build_report(movies, users, history, report_day), len(rejected))


run()

=== Daily Netflix Report: 2026-11-04 ===
Total viewing activity : 4 views, 310 minutes
Most watched movies    : Inception (2), Stranger Things (1), The Crown (1)
Active users (3)     : Aisha, Ben, Chen
Average ratings        : {'Inception': 4.5, 'Stranger Things': 4.0}
(Data quality: 3 invalid records rejected)


## 6. Inspect rejected records (data quality check)

In [7]:
_, _, clean, rejected = transform(*extract())
print(f"{len(clean)} clean, {len(rejected)} rejected")
for r in rejected:
    print(r)

5 clean, 3 rejected
{'user_id': 11, 'movie_id': 99, 'watched_on': '2026-11-04', 'minutes': 30, 'rating': 3}
{'user_id': 12, 'movie_id': 1, 'watched_on': '2026-11-04', 'minutes': -5, 'rating': 4}
{'user_id': 11, 'movie_id': 3, 'watched_on': '2026-11-04', 'minutes': 40, 'rating': 9}


## Notes
- Movies without ratings are excluded from the averages (not counted as 0).
- To automate daily: schedule `run()` with cron or Airflow and pass the date.
- Possible extensions: read from CSV, add logging of row counts, write the report to a file.